Permissions

In [0]:
%sql

GRANT READ VOLUME ON VOLUME workspace.default.datos_tacobueno TO `a01648461@tec.mx`;
GRANT SELECT ON TABLE workspace.default.ventas_taco_bueno_clean TO `a01648461@tec.mx`;

Read all CSV

In [0]:
# Leer todos los archivos CSV de la carpeta
ruta_volumen = "/Volumes/workspace/default/datos_tacobueno/Taco Bueno - Pricing/"

df_raw = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(ruta_volumen)
)

# Mostrar un vistazo de los datos y contar cuántas filas son
print(f"Total de registros: {df_raw.count():,}")
display(df_raw.limit(5))

Total de registros: 85,273,331


Store #,Date,Total,Check Number,Time Card,Begin Time,End Time,Total Discount,Order Voided,Tax,Quantity,Transaction Type,Menu Item,Menu Item Ext Ref,Pop,POS Code,Menu Price,Item Discount,Item Voided,Report Group,Tender Type
6905,5/4/2026,5.82,97148261638150,null,10:11 AM,10:12 AM,0.0,0.0,0.44,1,Regular Sale,Bean Burrito,null,Drive Thru,001063,2.69,0.0,0.0,(M) Burritos,null
6905,5/4/2026,5.82,97148261638150,null,10:11 AM,10:12 AM,0.0,0.0,0.44,1,Regular Sale,Beef Crispy Taco,null,Drive Thru,001018,2.69,0.0,0.0,(M) Tacos,null
6905,5/4/2026,5.82,97148261638150,null,10:11 AM,10:12 AM,0.0,0.0,0.44,1,Regular Sale,null,null,Drive Thru,NO 001000,0.0,0.0,0.0,null,null
6905,5/4/2026,5.82,97148261638150,null,10:11 AM,10:12 AM,0.0,0.0,0.44,1,Regular Sale,null,null,Drive Thru,NO 001002,0.0,0.0,0.0,null,null
6905,5/4/2026,32.49,97148263708674,null,10:24 AM,10:09 AM,0.0,0.0,2.48,1,Regular Sale,Beef Crispy Taco,null,Online: GrubHub,001018,3.49,0.0,0.0,(M) Tacos,null


Standarize columns name, data cleaning, create unified table

In [0]:
import re
from pyspark.sql.functions import col

# 1. Estandarizar nombres de columnas (eliminar espacios y el símbolo '#')
df_clean = df_raw
for columna in df_raw.columns:
  nombre_limpio = columna.strip().replace("#", "num").replace(" ", "_")
  nombre_limpio = re.sub(r"[^a-zA-Z0-9_]", "", nombre_limpio)
  df_clean = df_clean.withColumnRenamed(columna, nombre_limpio)

# 2. Convertir Check_Number a texto y filtrar cancelaciones
df_clean = (
    df_clean.withColumn("Check_Number", col("Check_Number").cast("string"))
    .filter(
        (col("Order_Voided").cast("double") == 0)
        | (col("Order_Voided").isNull())
    )
    .filter(
        (col("Item_Voided").cast("double") == 0)
        | (col("Item_Voided").isNull())
    )
)

# 3. Guardar como Tabla Delta oficial
df_clean.write.mode("overwrite").saveAsTable(
    "workspace.default.ventas_taco_bueno_clean"
)
print("¡Tabla creada con éxito! ")

¡Tabla creada con éxito! 


Analysis A: Top Revenue & Best-Selling Menu Items

In [0]:
%sql
SELECT 
  Menu_Item,
  SUM(CAST(Quantity AS DOUBLE)) AS Total_Units_Sold,
  ROUND(SUM(CAST(Quantity AS DOUBLE) * CAST(Menu_Price AS DOUBLE)), 2) AS Total_Gross_Revenue_USD
FROM workspace.default.ventas_taco_bueno_clean
GROUP BY Menu_Item
ORDER BY Total_Gross_Revenue_USD DESC
LIMIT 10;

Menu_Item,Total_Units_Sold,Total_Gross_Revenue_USD
Bean Burrito,4823017.0,1.246187825E7
Beef Crispy Taco,5756140.0,1.051855861E7
Party Burrito,7733862.0,1.039832703E7
Beef Muchaco,2691921.0,8578021.03
Beef Wholotta Platter 2025,716770.0,8079976.64
Chicken Quesadilla (w/sauce),949270.0,5746809.68
Combo 1 (NR),543948.0,5349514.02
Party Taco,4710271.0,5290637.44
Beef Soft Taco,1985751.0,4827902.85
Beef BOB,978077.0,4400460.25


Analysis B: Promotion Effectiveness (Discounted vs. Regular Price)

In [0]:
%sql
SELECT 
  CASE WHEN CAST(Item_Discount AS DOUBLE) > 0 THEN 'Discounted' ELSE 'Regular Price' END AS Pricing_Type,
  COUNT(DISTINCT Check_Number) AS Total_Orders,
  SUM(CAST(Quantity AS DOUBLE)) AS Total_Units_Sold,
  ROUND(AVG(CAST(Quantity AS DOUBLE)), 2) AS Avg_Units_Per_Line,
  ROUND(SUM(CAST(Item_Discount AS DOUBLE)), 2) AS Total_Discounts_Given_USD,
  ROUND(SUM(CAST(Quantity AS DOUBLE) * CAST(Menu_Price AS DOUBLE) - CAST(Item_Discount AS DOUBLE)), 2) AS Net_Revenue_USD
FROM workspace.default.ventas_taco_bueno_clean
GROUP BY CASE WHEN CAST(Item_Discount AS DOUBLE) > 0 THEN 'Discounted' ELSE 'Regular Price' END;

Pricing_Type,Total_Orders,Total_Units_Sold,Avg_Units_Per_Line,Total_Discounts_Given_USD,Net_Revenue_USD
Discounted,310863,2510417.0,1.55,3000226.86,5449650.14
Regular Price,6718081,8.9756346E7,1.08,-249.29,1.7797512031E8


Analysis C: Sales Channel Performance (Drive-Thru vs. Online / Dine-In)

In [0]:
%sql

SELECT 
  Pop AS Sales_Channel,
  COUNT(DISTINCT Check_Number) AS Total_Orders,
  SUM(CAST(Quantity AS DOUBLE)) AS Units_Sold,
  ROUND(SUM(CAST(Total AS DOUBLE)), 2) AS Total_Sales_USD
FROM workspace.default.ventas_taco_bueno_clean
GROUP BY Pop
ORDER BY Total_Sales_USD DESC;


Sales_Channel,Total_Orders,Units_Sold,Total_Sales_USD
Drive Thru,4167347,2.7521215E7,5.2005306681E8
To Go,366623,3.1262902E7,4.3201522139E8
Inside,274558,1.3505856E7,2.4446554119E8
Dine-In,1122057,7549436.0,2.0211889323E8
Online: DoorDash,310855,3398845.0,1.0302077082E8
Carry Out,470694,3132560.0,7.753568578E7
3,187462,4363995.0,7.443273566E7
Online: UberEats,87165,1029922.0,3.405367812E7
Online,27319,383994.0,1.502486462E7
Online: GrubHub,8666,99126.0,3354218.42


Analysis D: The "Lift" Test (Do Promos Increase Overall Ticket Size?)

In [0]:
%sql
SELECT 
  CASE WHEN Max_Discount > 0 THEN 'Checks WITH Promos' ELSE 'Checks WITHOUT Promos' END AS Ticket_Type,
  COUNT(DISTINCT Check_Number) AS Total_Customers,
  ROUND(SUM(Check_Total_Sales), 2) AS Total_Revenue_USD,
  ROUND(SUM(Check_Total_Sales) / COUNT(DISTINCT Check_Number), 2) AS Average_Basket_Size_USD
FROM (
  SELECT 
    Check_Number,
    SUM(CAST(Total AS DOUBLE)) AS Check_Total_Sales,
    MAX(CAST(Item_Discount AS DOUBLE)) AS Max_Discount
  FROM workspace.default.ventas_taco_bueno_clean
  GROUP BY Check_Number
)
GROUP BY CASE WHEN Max_Discount > 0 THEN 'Checks WITH Promos' ELSE 'Checks WITHOUT Promos' END;

Ticket_Type,Total_Customers,Total_Revenue_USD,Average_Basket_Size_USD
Checks WITHOUT Promos,6408307,1.10211455414E9,171.98
Checks WITH Promos,310863,6.0443309297E8,1944.37


Analysis E: Cannibalization

In [0]:
%sql
SELECT 
  Menu_Item,
  SUM(CAST(Quantity AS DOUBLE)) AS Total_Units_Sold,
  SUM(CASE WHEN CAST(Item_Discount AS DOUBLE) > 0 THEN CAST(Quantity AS DOUBLE) ELSE 0 END) AS Units_Sold_On_Discount,
  SUM(CASE WHEN CAST(Item_Discount AS DOUBLE) = 0 THEN CAST(Quantity AS DOUBLE) ELSE 0 END) AS Units_Sold_Regular_Price,
  ROUND((SUM(CASE WHEN CAST(Item_Discount AS DOUBLE) > 0 THEN CAST(Quantity AS DOUBLE) ELSE 0 END) / NULLIF(SUM(CAST(Quantity AS DOUBLE)), 0)) * 100, 2) AS Discount_Dependency_Percentage
FROM workspace.default.ventas_taco_bueno_clean
GROUP BY Menu_Item
HAVING Total_Units_Sold > 1000 -- Focus only on popular items
ORDER BY Discount_Dependency_Percentage DESC
LIMIT 15;

Menu_Item,Total_Units_Sold,Units_Sold_On_Discount,Units_Sold_Regular_Price,Discount_Dependency_Percentage
Chicken Soft Taco Bar (delivery),1828.0,1791.0,37.0,97.98
Beef Crispy Taco Bar (delivery),2156.0,2064.0,92.0,95.73
Beef Soft Taco Bar (delivery),1633.0,1550.0,83.0,94.92
Steak Soft Taco,173713.0,92851.0,80862.0,53.45
Brisket (Cond.),1374.0,652.0,722.0,47.45
Blended Cheese - Side,2537.0,1126.0,1411.0,44.38
Chicken - Side,34553.0,12077.0,22476.0,34.95
Lettuce - Side,12751.0,3583.0,9168.0,28.1
Blended Cheese (Cond.),1498.0,410.0,1088.0,27.37
Steak (Cond.),1083.0,274.0,809.0,25.3


Analysis F: Seasonality & Peak Hours

In [0]:
%sql
SELECT 
  SUBSTRING(Begin_Time, 1, INSTR(Begin_Time, ':') - 1) AS Hour_Of_Day,
  RIGHT(Begin_Time, 2) AS AM_PM,
  COUNT(DISTINCT Check_Number) AS Total_Orders,
  ROUND(SUM(Total), 2) AS Total_Sales_USD
FROM workspace.default.ventas_taco_bueno_clean
WHERE Begin_Time IS NOT NULL
GROUP BY 
  SUBSTRING(Begin_Time, 1, INSTR(Begin_Time, ':') - 1),
  RIGHT(Begin_Time, 2)
ORDER BY AM_PM ASC, CAST(Hour_Of_Day AS INT) ASC;

Hour_Of_Day,AM_PM,Total_Orders,Total_Sales_USD
1,AM,2042,214263.73
2,AM,669,236162.68
3,AM,382,118275.18
4,AM,434,147761.38
5,AM,789,128246.79
6,AM,1689,146560.4
7,AM,2474,430972.98
8,AM,5934,1809596.22
9,AM,271092,4.355007001E7
10,AM,1017489,1.6345550403E8


Analysis G: Halo Effect / Attach Rate

In [0]:
%sql
WITH base AS (
  SELECT
    Check_Number,
    Menu_Item,
    COALESCE(TRY_CAST(Item_Discount AS DOUBLE), 0) AS Discount_Num
  FROM workspace.default.ventas_taco_bueno_clean
),
top_promo_items AS (
  SELECT Menu_Item AS Promo_Item
  FROM base
  WHERE Discount_Num > 0
  GROUP BY Menu_Item
  ORDER BY COUNT(DISTINCT Check_Number) DESC
  LIMIT 5
),
promo_checks AS (
  SELECT DISTINCT b.Check_Number, b.Menu_Item AS Promo_Item
  FROM base b
  JOIN top_promo_items p ON b.Menu_Item = p.Promo_Item
  WHERE b.Discount_Num > 0
),
companions AS (
  SELECT
    pc.Promo_Item,
    b.Menu_Item AS Companion_Item,
    COUNT(DISTINCT b.Check_Number) AS Checks_Together
  FROM promo_checks pc
  JOIN base b ON b.Check_Number = pc.Check_Number
  WHERE b.Menu_Item <> pc.Promo_Item
  GROUP BY pc.Promo_Item, b.Menu_Item
)
SELECT *
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY Promo_Item ORDER BY Checks_Together DESC) AS Rank_Num
  FROM companions
)
WHERE Rank_Num <= 3
ORDER BY Promo_Item, Rank_Num;

Promo_Item,Companion_Item,Checks_Together,Rank_Num
Bean Burrito,Party Burrito,96463,1
Bean Burrito,Modifier (NR),95554,2
Bean Burrito,Beef Crispy Taco,95360,3
Beef Crispy Taco,Modifier (NR),120250,1
Beef Crispy Taco,Party Burrito,119060,2
Beef Crispy Taco,Bean Burrito,117241,3
Beef Muchaco,Modifier (NR),58170,1
Beef Muchaco,Party Burrito,57885,2
Beef Muchaco,Bean Burrito,57410,3
Party Burrito,Modifier (NR),64169,1


H. Promo ROI / Discount Efficiency

In [0]:
%sql
SELECT
  Menu_Item,
  ROUND(SUM(CAST(Total AS DOUBLE)), 2) AS Revenue_USD,
  ROUND(SUM(CAST(Item_Discount AS DOUBLE)), 2) AS Discount_USD,
  ROUND(SUM(CAST(Total AS DOUBLE)) / NULLIF(SUM(CAST(Item_Discount AS DOUBLE)), 0), 2) AS Revenue_per_Discount_Dollar
FROM workspace.default.ventas_taco_bueno_clean
GROUP BY Menu_Item
HAVING Discount_USD > 0
ORDER BY Revenue_per_Discount_Dollar ASC
LIMIT 15;

Menu_Item,Revenue_USD,Discount_USD,Revenue_per_Discount_Dollar
Diced Chicken (1 pound),2463.83,1217.13,2.02
Brisket - Side,6919.59,1200.53,5.76
Breakfast Box (NR),6540.45,1127.2,5.8
Taco Meat (1 pound),3545.19,492.78,7.19
Mexican Rice (1 gallon),3668.66,415.21,8.84
Chicken Soft Taco Bar (delivery),58852.38,6538.46,9.0
Beef Crispy Taco Bar (delivery),69818.43,6929.93,10.07
Mineragua,3658.33,333.06,10.98
Steak Soft Taco,2658369.12,239285.41,11.11
Sidral Mundet,2306.94,200.33,11.52
